 # Manually recalculating/refitting AVAR

In [1]:
import gc
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = 'updated-data/obs/lme-ready'

In [3]:
def add_avar_col(df, val_col, merge_cols):
    # add next step in series
    df[val_col + '_2'] = None
    df[val_col + '_3'] = None

    groups = df.groupby(by=merge_cols)
    completed = []
    for _, dfi in groups:
        idx = dfi.index
        dfi[val_col + '_2'].loc[idx[:-1]] = dfi[val_col].values[1:]
        dfi[val_col + '_3'].loc[idx[:-2]] = dfi[val_col].values[2:]
        dfi['AVAR'] = (1 / (2 * (len(dfi) ** 2))) * (dfi[val_col + '_3'] - (2 * dfi[val_col + '_2']) + dfi[val_col]) ** 2
        completed+=[dfi]
    df = pd.concat(completed, ignore_index=True)

    del df[val_col + '_2']
    del df[val_col + '_3']
    gc.collect()

    return df

## Showtime

In [4]:
fs = [f for f in os.listdir(DATA_PATH) if f.endswith('.parquet') and (not f.startswith('.'))]
len(fs)

5591

In [5]:
val_col = 'I'
turn_col = 'x_turn_id'
conversation_id_col = 'conversation_id'

# origin/correct
# merge_cols = [conversation_id_col, turn_col, 'self']

##### Experiments
merge_cols = [conversation_id_col, turn_col, 'y_speaker']
# merge_cols = [conversation_id_col, turn_col]

In [6]:
for f in tqdm(fs[4300:]):
    f_ = os.path.join(DATA_PATH, f)
    df = pd.read_parquet(f_, engine='fastparquet')

    df = df.sort_values(by=['x_turn_id', 'y_turn_id'])
    df.index = range(len(df))

    df['self'] = df['x_speaker'] == df['y_speaker']

    df = add_avar_col(df, val_col=val_col, merge_cols=merge_cols)

    # if saving AVAR for later, do:
    df.to_parquet(
        f_,
        engine='fastparquet',
        compression='snappy'
    )

  0%|          | 0/1291 [00:00<?, ?it/s]